In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from ipywidgets import interact_manual

cwd = Path.cwd()
if (cwd / 'wigner_levy.py').exists():
    sys.path.insert(0, str(cwd))
elif (cwd / 'random_dnn' / 'wigner_levy.py').exists():
    sys.path.insert(0, str(cwd / 'random_dnn'))

import wigner_levy as cb

In [ ]:
DEFAULT_PARAMS = dict(
    mu=1.5,
    entry_scale=1.0,
    matrix_size=64,
    num_matrices=8,
    bins=81,
    z_prime_max=12.0,
    num_points=161,
    integration_method='quad',
    mc_samples=20000,
    mc_seed=0,
    seed=0,
    overlay_schemes=True,
    show_asymptotic_tail=True,
    tail_start=3.0,
)

DEFAULT_PARAMS

In [ ]:
def plot_density_comparison(
    mu,
    entry_scale,
    matrix_size,
    num_matrices,
    bins,
    z_prime_max,
    num_points,
    mc_samples,
    mc_seed,
    seed,
    show_empirical=True,
    show_quad_curve=True,
    show_monte_carlo_curve=True,
    show_asymptotic_tail=False,
    tail_start=3.0,
):
    params = dict(
        mu=float(mu),
        entry_scale=float(entry_scale),
        matrix_size=int(matrix_size),
        num_matrices=int(num_matrices),
        bins=int(bins),
        z_prime_max=float(z_prime_max),
        num_points=int(num_points),
        mc_samples=int(mc_samples),
        mc_seed=int(mc_seed),
        seed=int(seed),
    )

    empirical = (
        cb.empirical_wigner_levy_spectrum(
            params['mu'],
            matrix_size=params['matrix_size'],
            num_matrices=params['num_matrices'],
            entry_scale=params['entry_scale'],
            bins=params['bins'],
            seed=params['seed'],
        )
        if show_empirical else None
    )

    curves = {}
    if show_quad_curve:
        curves['quad'] = cb.theoretical_wigner_levy_curve(
            params['mu'],
            entry_scale=params['entry_scale'],
            z_prime_max=params['z_prime_max'],
            num_points=params['num_points'],
            integration_method='quad',
            mc_samples=params['mc_samples'],
            mc_seed=params['mc_seed'],
        )
    if show_monte_carlo_curve:
        curves['monte_carlo'] = cb.theoretical_wigner_levy_curve(
            params['mu'],
            entry_scale=params['entry_scale'],
            z_prime_max=params['z_prime_max'],
            num_points=params['num_points'],
            integration_method='monte_carlo',
            mc_samples=params['mc_samples'],
            mc_seed=params['mc_seed'],
        )

    ordered_methods = [method for method in ('quad', 'monte_carlo') if method in curves]
    color_map = {'quad': 'black', 'monte_carlo': 'darkorange'}
    marker_map = {'quad': 'o', 'monte_carlo': '+'}
    ms_map = {'quad': 4, 'monte_carlo': 6}
    palette = {
        'quad':        {'C': 'tab:blue',       'beta': 'tab:green'},
        'monte_carlo': {'C': 'cornflowerblue', 'beta': 'mediumseagreen'},
    }

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

    if show_empirical and empirical is not None:
        bin_widths = np.diff(empirical.bin_edges)
        axes[0].bar(
            empirical.bin_centers,
            empirical.density,
            width=bin_widths,
            color='steelblue',
            alpha=0.35,
            label=f'exact diagonalization (N={empirical.matrix_size}, n={empirical.num_matrices})',
        )
    for method in ordered_methods:
        curve = curves[method]
        axes[0].plot(
            curve.lambda_grid,
            curve.density,
            color=color_map[method],
            ls='-',
            marker='.',
            ms=4,
            label=f'{method} mean-field theory',
        )
    axes[0].axhline(
        cb.mean_field_density_at_zero(params['mu'], entry_scale=params['entry_scale']),
        color='gray',
        ls=':',
        lw=1,
        label=r'$\rho(0)$ mean-field',
    )

    lambda_limit = (
        max(np.nanmax(np.abs(curve.lambda_grid)) for curve in curves.values())
        if curves else float(params['z_prime_max'])
    )
    tail_prefactor = cb.dos_tail_prefactor(params['mu'], entry_scale=params['entry_scale'])
    if show_asymptotic_tail:
        tail_grid = np.linspace(-lambda_limit, lambda_limit, 801)
        tail_density = cb.asymptotic_dos_tail(tail_grid, params['mu'], entry_scale=params['entry_scale'])
        tail_mask = np.abs(tail_grid) >= float(tail_start)
        axes[0].plot(
            tail_grid[tail_mask],
            tail_density[tail_mask],
            color='crimson',
            ls=':',
            lw=2,
            label=rf'mean-field asymptotic tail ${tail_prefactor:.4f}|\lambda|^{{-(1 + {params["mu"]:.2f})}}$',
        )

    axes[0].set_xlabel(r'$\lambda$')
    axes[0].set_ylabel(r'$\rho(\lambda)$')
    axes[0].set_title(rf'$\mu={params["mu"]}$, entry scale = {params["entry_scale"]}')
    axes[0].set_xlim(-lambda_limit, lambda_limit)
    axes[0].legend()

    for method in ordered_methods:
        curve = curves[method]
        marker = marker_map[method]
        ms = ms_map[method]
        colors = palette[method]
        axes[1].plot(
            curve.lambda_grid,
            curve.c_eff,
            color=colors['C'],
            ls='-',
            marker=marker,
            ms=ms,
            label=rf'$C(\lambda)$ {method}',
        )
        axes[1].plot(
            curve.lambda_grid,
            curve.beta_eff,
            color=colors['beta'],
            ls='-',
            marker=marker,
            ms=ms,
            label=rf'$\beta(\lambda)$ {method}',
        )
    if show_asymptotic_tail:
        tail_curve = cb.asymptotic_tail_curve(
            params['mu'],
            entry_scale=params['entry_scale'],
            z_prime_max=params['z_prime_max'],
            num_points=params['num_points'],
        )
        axes[1].plot(
            tail_curve.lambda_grid,
            tail_curve.c_eff,
            color='crimson',
            ls=':',
            lw=2,
            label=r'$C(\lambda)$ mean-field asymptotic tail',
        )
        axes[1].plot(
            tail_curve.lambda_grid,
            tail_curve.beta_eff,
            color='crimson',
            ls='--',
            lw=1.5,
            label=r'$\beta(\lambda)$ mean-field asymptotic tail',
        )
    axes[1].set_xlabel(r'$\lambda$')
    axes[1].set_title('Auxiliary mean-field parameters')
    axes[1].legend()

    plt.tight_layout()
    plt.show()

    rho0_formula = cb.mean_field_density_at_zero(params['mu'], entry_scale=params['entry_scale'])
    print(f'mean-field rho(0) formula={rho0_formula:.6f}')
    print(f'mean-field asymptotic tail: rho(lambda) ~ {tail_prefactor:.6f} / |lambda|^(1 + {params["mu"]:.3f})')
    for method in ordered_methods:
        curve = curves[method]
        rho0_curve = curve.density[len(curve.density) // 2]
        message = (
            f"{method}: lambda range [{np.nanmin(curve.lambda_grid):.3f}, {np.nanmax(curve.lambda_grid):.3f}] | "
            f"mean-field rho(0)={rho0_curve:.6f}"
        )
        if method == 'monte_carlo':
            message += f" | mc_samples={curve.mc_samples}, mc_seed={curve.mc_seed}"
        print(message)

In [ ]:
# plot_density_comparison(**DEFAULT_PARAMS)

In [ ]:
import traitlets


class ToggleableGroup(widgets.VBox, widgets.ValueWidget):
    """VBox with a checkbox toggle; .value is {'enabled': bool, **{name: child.value, ...}}."""

    value = traitlets.Dict({})

    def __init__(self, label, params, enabled=True, **kwargs):
        self._params = params

        self.toggle = widgets.Checkbox(
            value=enabled,
            description=label,
            indent=False,
            style={'description_width': 'initial'},
        )
        children_box = widgets.VBox(
            list(params.values()),
            layout=widgets.Layout(padding='2px 0 2px 14px'),
        )
        super().__init__(
            children=[self.toggle, children_box],
            layout=widgets.Layout(border='1px solid #ccc', padding='6px 8px', margin='0 4px 4px 0'),
            **kwargs,
        )
        self.toggle.observe(self._on_change, names='value')
        for w in params.values():
            w.observe(self._on_change, names='value')
        self._sync()

    def _sync(self):
        enabled = self.toggle.value
        for w in self._params.values():
            w.disabled = not enabled
        self.value = {'enabled': enabled, **{k: w.value for k, w in self._params.items()}}

    def _on_change(self, _):
        self._sync()


_child = lambda desc, cls, val: cls(value=val, description=desc, style={'description_width': 'initial'})

empirical_group = ToggleableGroup('Empirical (exact diagonalization)', {
    'matrix_size':  _child('matrix_size',  widgets.IntText,   DEFAULT_PARAMS['matrix_size']),
    'num_matrices': _child('num_matrices', widgets.IntText,   DEFAULT_PARAMS['num_matrices']),
    'bins':         _child('bins',         widgets.IntText,   DEFAULT_PARAMS['bins']),
    'seed':         _child('seed',         widgets.IntText,   DEFAULT_PARAMS['seed']),
})

quad_group = ToggleableGroup('Quad integration', {})

mc_group = ToggleableGroup('MC integration', {
    'mc_samples': _child('mc_samples', widgets.IntText,   DEFAULT_PARAMS['mc_samples']),
    'mc_seed':    _child('mc_seed',    widgets.IntText,   DEFAULT_PARAMS['mc_seed']),
})

tail_group = ToggleableGroup('Mean-field asymptotic tail', {
    'tail_start': _child('tail_start', widgets.FloatText, DEFAULT_PARAMS['tail_start']),
})


@interact_manual(
    mu=widgets.FloatText(value=DEFAULT_PARAMS['mu']),
    entry_scale=widgets.FloatText(value=DEFAULT_PARAMS['entry_scale']),
    z_prime_max=widgets.FloatText(value=DEFAULT_PARAMS['z_prime_max']),
    num_points=widgets.IntText(value=DEFAULT_PARAMS['num_points']),
    empirical=empirical_group,
    quad=quad_group,
    monte_carlo=mc_group,
    tail=tail_group,
)
def explore_density(mu, entry_scale, z_prime_max, num_points, empirical, quad, monte_carlo, tail):
    plot_density_comparison(
        mu=mu,
        entry_scale=entry_scale,
        matrix_size=empirical['matrix_size'],
        num_matrices=empirical['num_matrices'],
        bins=empirical['bins'],
        z_prime_max=z_prime_max,
        num_points=num_points,
        mc_samples=monte_carlo['mc_samples'],
        mc_seed=monte_carlo['mc_seed'],
        seed=empirical['seed'],
        show_empirical=empirical['enabled'],
        show_quad_curve=quad['enabled'],
        show_monte_carlo_curve=monte_carlo['enabled'],
        show_asymptotic_tail=tail['enabled'],
        tail_start=tail['tail_start'],
    )